# Train a robot controller

This notebook runs either the VAE–LSTM–MDN or CNN–MLP staged, resumable recipe selected below. Re-running the training cell resumes at the latest completed epoch. The graph and stage timeline refresh after every epoch.

In [ ]:
import json
from pathlib import Path
import sys

sys.path.append("..")
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from robot_controller.trec_factory import create_training_recipe
from robot_controller.visualize_trec import TrainingRecipeVisualizer

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
experiment_name = "robot_controller_training"
run_name = "trec_vae_neo_lstm_mdn_sample"
# For the deterministic CNN--MLP architecture, use:
# run_name = "trec_cnn_mlp_sample"

if expruns_path:
    expruns_path = Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    for name in [experiment_name, "robot_controller", "sensorprocessing_conv_vae_neo", "sensorprocessing_propriotuned_cnn", "demonstration", "robot_al5d"]:
        Config().copy_experiment(name)
if results_path:
    results_path = Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment_name, run_name, creation_style=creation_style)
recipe = create_training_recipe(exp)

In [ ]:
display(TrainingRecipeVisualizer(exp).build())
print(json.dumps(recipe.status, indent=2))

In [ ]:
def plot_metrics(data_dir):
    path = Path(data_dir) / "metrics.jsonl"
    if not path.is_file():
        return
    records = [json.loads(line) for line in path.read_text().splitlines() if line]
    if not records:
        return
    _, ax = plt.subplots(figsize=(10, 4))
    for stage in dict.fromkeys(record["stage"] for record in records):
        selected = [record for record in records if record["stage"] == stage]
        validation_key = "validation_nll" if "validation_nll" in selected[0] else "validation_loss"
        training_key = "train_nll" if "train_nll" in selected[0] else "train_loss"
        ax.plot([record["epoch"] for record in selected], [record[validation_key] for record in selected], marker=".", label=f"{stage}: validation")
        ax.plot([record["epoch"] for record in selected], [record[training_key] for record in selected], linestyle="--", label=f"{stage}: training")
    ax.set(xlabel="Stage epoch", ylabel="Training objective")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

def refresh(status):
    clear_output(wait=True)
    display(TrainingRecipeVisualizer(exp, status=status).build())
    plot_metrics(exp.data_dir())
    print(f"state={status['state']} stage={status.get('current_stage')} epoch={status.get('epoch')}/{status.get('epochs')}")

In [ ]:
# KeyboardInterrupt is recorded as an interrupted state and then re-raised.
# Run this cell again to resume from the latest completed epoch.
trained_model = recipe.train(progress_callback=refresh)

In [ ]:
refresh(recipe.status)
print(f"Deployable bundle: {exp.data_dir() / exp['model_file']}")